# Tunisian Bac Math RAG — Scientific Validation

**Purpose:** End-to-end validation of the RAG pipeline for thesis documentation.  
**What this tests:** environment integrity, DB contents, two-stage retrieval quality, LLM generation correctness, syllabus guard.  
**What this does NOT do:** re-index, rebuild, or modify any data.  

---

## 0. Environment & Sanity Checks

Verify that the runtime has everything needed before loading heavy models.

In [ ]:
import sys
import os
import time
from pathlib import Path

print(f"Python  : {sys.version}")
print(f"Platform: {sys.platform}")
print(f"CWD     : {os.getcwd()}")
print()

In [ ]:
# ── Package import checks ────────────────────────────────────
_ok = []
_fail = []

for pkg_name, import_name in [
    ("chromadb",       "chromadb"),
    ("FlagEmbedding",  "FlagEmbedding"),
    ("torch",          "torch"),
    ("vertexai",       "vertexai"),
    ("google.cloud.storage", "google.cloud.storage"),
    ("streamlit",      "streamlit"),
]:
    try:
        mod = __import__(import_name)
        ver = getattr(mod, "__version__", "(no __version__)")
        _ok.append((pkg_name, ver))
    except ImportError as e:
        _fail.append((pkg_name, str(e)))

for name, ver in _ok:
    print(f"  OK   {name:30s} {ver}")
for name, err in _fail:
    print(f"  FAIL {name:30s} {err}")

if _fail:
    print("\n!! Some packages are missing. Install them before continuing.")
else:
    print(f"\nAll {len(_ok)} packages imported successfully.")

In [ ]:
# ── Project files & DB path ──────────────────────────────────
PROJECT_ROOT = Path(os.getcwd())

required_files = ["config.py", "rag_engine.py"]
for f in required_files:
    p = PROJECT_ROOT / f
    status = "EXISTS" if p.exists() else "MISSING"
    print(f"  {status:7s} {f}")

db_path = PROJECT_ROOT / "tunisian_math_db"
print(f"\n  DB dir exists : {db_path.exists()}")
if db_path.exists():
    items = list(db_path.iterdir())
    print(f"  DB dir items  : {len(items)}")
    for item in sorted(items)[:8]:
        print(f"    {item.name}")

In [ ]:
# ── Confirm config loads and show key values ─────────────────
import config

print("Config loaded successfully.")
print(f"  PROJECT_ID                : {config.PROJECT_ID}")
print(f"  REGION                    : {config.REGION}")
print(f"  LOCAL_DB_PATH             : {config.LOCAL_DB_PATH}")
print(f"  COLLECTION_NAME           : {config.COLLECTION_NAME}")
print(f"  EMBEDDING_MODEL_NAME      : {config.EMBEDDING_MODEL_NAME}")
print(f"  USE_FP16                  : {config.USE_FP16}")
print(f"  CHAT_MODEL_ID             : {config.CHAT_MODEL_ID}")
print(f"  RETRIEVE_K_FIRST_PASS     : {config.RETRIEVE_K_FIRST_PASS}")
print(f"  RETRIEVE_K_SECOND_PASS    : {config.RETRIEVE_K_SECOND_PASS}")
print(f"  USE_TOP_N                 : {config.USE_TOP_N}")
print(f"  SIMILARITY_GOOD_THRESHOLD : {config.SIMILARITY_GOOD_THRESHOLD}")
print(f"  SIMILARITY_FALLBACK_THRESH: {config.SIMILARITY_FALLBACK_THRESHOLD}")
print(f"  CHUNK_CORRECTION          : {config.CHUNK_CORRECTION}")
print(f"  CHUNK_TEXTBOOK            : {config.CHUNK_TEXTBOOK}")

---
## 1. Initialize RAG Engine

Loads BGE-M3 embeddings, ChromaDB collection, and Vertex AI model.  
Expect **30-90 s** on first run (model download / warm-up).

In [ ]:
from rag_engine import TunisianMathRAG, QueryResult, RetrievedDoc

t0 = time.monotonic()
engine = TunisianMathRAG()
init_time = time.monotonic() - t0

print(f"\nEngine ready in {init_time:.1f}s")
print(f"Collection : {config.COLLECTION_NAME}")
print(f"Chunks     : {engine.chunk_count:,}")

---
## 2. Database Metadata Profile (FREE — no API cost)

Sample the collection to understand what is indexed:  
distribution by `type`, `chapter`, `is_solution`.

In [ ]:
from collections import Counter

# ChromaDB .get() with limit; grab a meaningful sample
SAMPLE_SIZE = min(2000, engine.chunk_count)
sample = engine.collection.get(limit=SAMPLE_SIZE, include=["metadatas"])
metas = sample["metadatas"] or []

print(f"Sampled {len(metas):,} chunks out of {engine.chunk_count:,} total.\n")

type_dist    = Counter(m.get("type", "??") for m in metas)
sol_dist     = Counter(m.get("is_solution", "??") for m in metas)
chapter_dist = Counter(m.get("chapter", "??") for m in metas)
year_dist    = Counter(m.get("year", "") for m in metas)

print("=== Document type distribution ===")
for t, c in type_dist.most_common():
    pct = 100 * c / len(metas)
    bar = '#' * int(pct / 2)
    print(f"  {t:18s} {c:5d}  ({pct:5.1f}%)  {bar}")

print("\n=== is_solution distribution ===")
for s, c in sol_dist.most_common():
    pct = 100 * c / len(metas)
    print(f"  {s:18s} {c:5d}  ({pct:5.1f}%)")

print("\n=== Chapter distribution (top 15) ===")
for ch, c in chapter_dist.most_common(15):
    pct = 100 * c / len(metas)
    print(f"  {ch:35s} {c:5d}  ({pct:5.1f}%)")

# Years with content
years_with_data = sorted([y for y in year_dist if y], key=lambda y: int(y) if y.isdigit() else 0)
print(f"\n=== Years with data ({len(years_with_data)}) ===")
if years_with_data:
    print(f"  Range: {years_with_data[0]} — {years_with_data[-1]}")
    print(f"  All  : {', '.join(years_with_data)}")

---
## 3. Retrieval-Only Tests (FREE — no Gemini API cost)

These call `engine._two_stage_retrieve()` which runs:
1. **First pass**: corrections / bac / series with `is_solution=true`
2. **Second pass** (if needed): textbook / cours
3. **Unfiltered fallback** (if both empty)

This uses only local embeddings + ChromaDB. **Zero API cost.**  
Use this to validate retrieval quality and tune thresholds.

In [ ]:
# ── Test query bank ──────────────────────────────────────────
# Covers the main chapters of the Tunisian Bac Math curriculum.
# Each tuple: (query, expected_chapter_hint, description)

RETRIEVAL_QUERIES = [
    # Nombres complexes
    ("Déterminer le module et un argument de z = 1 + i\u221A3",
     "Nombres complexes", "Module & argument — core complex numbers"),

    # Suites numériques
    ("Montrer que la suite (Un) définie par Un+1 = f(Un) est convergente",
     "Suites", "Convergence of recursive sequence"),

    # Limites et continuité
    ("Calculer la limite de (sin x)/x quand x tend vers 0",
     "Limites", "Classic limit — sinx/x"),

    # Dérivabilité
    ("Étudier la dérivabilité de f en 0 et interpréter graphiquement",
     "Dérivabilité", "Differentiability study at a point"),

    # Intégration
    ("Calculer l'intégrale de 0 à pi/2 de x sin(x) dx par intégration par parties",
     "Intégration", "Integration by parts"),

    # Probabilités
    ("On lance un dé équilibré 3 fois. Calculer la probabilité d'obtenir exactement 2 six.",
     "Probabilités", "Binomial probability"),

    # Arithmétique
    ("Montrer que pour tout n entier naturel, n(n+1)(n+2) est divisible par 6",
     "Arithmétique", "Divisibility proof"),

    # Équations différentielles
    ("Résoudre l'équation différentielle y' + 2y = e^(-x)",
     "Équations différentielles", "First-order linear ODE"),

    # Géométrie dans l'espace
    ("Déterminer une équation cartésienne du plan passant par A, B et C",
     "Géométrie", "Cartesian equation of a plane"),

    # Logarithme / Exponentielle
    ("Résoudre dans R l'équation ln(x+1) + ln(x-1) = ln(3)",
     "Logarithme", "Logarithmic equation"),
]

In [ ]:
# ── Run retrieval-only on all queries ────────────────────────

retrieval_results = []

for i, (query, expected_ch, desc) in enumerate(RETRIEVAL_QUERIES, 1):
    print(f"\n{'='*75}")
    print(f"  QUERY {i}/{len(RETRIEVAL_QUERIES)}: {desc}")
    print(f"  Q: {query}")
    print(f"  Expected chapter hint: {expected_ch}")
    print(f"{'='*75}")

    t0 = time.monotonic()
    selected, first_pass, second_pass, case = engine._two_stage_retrieve(query)
    elapsed = time.monotonic() - t0

    # Build context to compute confidence
    ctx = engine._build_context(selected)
    confidence = engine._confidence_level(selected, ctx)

    best_dist = selected[0].distance if selected else None
    top_chapter = selected[0].metadata.get("chapter", "") if selected else ""
    top_type = selected[0].metadata.get("type", "") if selected else ""

    print(f"\n  Case         : {case}")
    print(f"  Confidence   : {confidence}")
    print(f"  First pass   : {len(first_pass)} docs")
    print(f"  Second pass  : {len(second_pass)} docs")
    print(f"  Selected     : {len(selected)} docs")
    print(f"  Best distance: {best_dist}")
    print(f"  Top chapter  : {top_chapter}")
    print(f"  Time         : {elapsed:.3f}s")

    print(f"\n  Selected documents:")
    for d in selected:
        m = d.metadata
        print(f"    Rank {d.rank:2d} | dist={d.distance:.4f} | "
              f"type={m.get('type',''):15s} | "
              f"ch={m.get('chapter',''):25s} | "
              f"yr={m.get('year',''):4s} | "
              f"sol={m.get('is_solution',''):5s} | "
              f"file={m.get('filename','')}")
        # Show first 120 chars of content
        snippet = d.content[:120].replace('\n', ' ')
        print(f"           Snippet: {snippet}...")

    # Accumulate for summary table
    retrieval_results.append({
        "idx": i,
        "description": desc,
        "query": query,
        "expected_chapter": expected_ch,
        "case": case,
        "confidence": confidence,
        "n_selected": len(selected),
        "n_first_pass": len(first_pass),
        "n_second_pass": len(second_pass),
        "best_distance": round(best_dist, 4) if best_dist is not None else None,
        "top_chapter": top_chapter,
        "top_type": top_type,
        "retrieval_time_s": round(elapsed, 3),
    })

print(f"\n\nRetrieval-only tests complete: {len(retrieval_results)} queries processed.")

### 3.1 Retrieval Summary Table (thesis-ready)

Compact view of all retrieval results.  
Screenshot this for your thesis report.

In [ ]:
# ── Summary table ────────────────────────────────────────────

hdr = (f"{'#':>2s}  {'Description':<30s}  {'Case':>4s}  {'Conf':>6s}  "
       f"{'Docs':>4s}  {'BestD':>7s}  {'TopChapter':<25s}  {'TopType':<15s}  {'Time':>6s}")
sep = '-' * len(hdr)

print(sep)
print(hdr)
print(sep)

for r in retrieval_results:
    desc_short = r['description'][:28] + '..' if len(r['description']) > 30 else r['description']
    bd = f"{r['best_distance']:.4f}" if r['best_distance'] is not None else '  N/A '
    print(f"{r['idx']:2d}  {desc_short:<30s}  {r['case']:>4s}  {r['confidence']:>6s}  "
          f"{r['n_selected']:4d}  {bd:>7s}  {r['top_chapter']:<25s}  {r['top_type']:<15s}  "
          f"{r['retrieval_time_s']:6.3f}")

print(sep)

# Aggregate stats
avg_time = sum(r['retrieval_time_s'] for r in retrieval_results) / len(retrieval_results)
case_a = sum(1 for r in retrieval_results if r['case'] == 'A')
case_b = sum(1 for r in retrieval_results if r['case'] == 'B')
dists = [r['best_distance'] for r in retrieval_results if r['best_distance'] is not None]
avg_dist = sum(dists) / len(dists) if dists else 0

print(f"\nAggregate:")
print(f"  Case A (correction match) : {case_a}/{len(retrieval_results)}")
print(f"  Case B (textbook fallback): {case_b}/{len(retrieval_results)}")
print(f"  Avg best distance         : {avg_dist:.4f}")
print(f"  Avg retrieval time        : {avg_time:.3f}s")

---
## 4. Full RAG Queries (Gemini API cost — targeted subset)

Runs `engine.query()` which calls Vertex AI Gemini.  
We select **4 representative queries** to validate the full pipeline  
(retrieval + prompt compilation + generation) while keeping API cost low.

Each result captures: answer text, retrieval case, confidence, timings, sources.

In [ ]:
# ── Full RAG test queries ────────────────────────────────────
# Deliberately small set to control API cost.
# Mix of modes: correction (dry, exam-style) and coaching (pedagogical).

RAG_QUERIES = [
    {
        "question": "Résoudre dans C l'équation z^2 + 2z + 5 = 0",
        "mode": "correction",
        "desc": "Complex equation — correction mode",
    },
    {
        "question": "Comment montrer qu'une suite définie par récurrence est convergente ?",
        "mode": "coaching",
        "desc": "Recursive sequence convergence — coaching mode",
    },
    {
        "question": "Calculer l'intégrale de 0 à 1 de x*e^x dx",
        "mode": "correction",
        "desc": "Integration by parts — correction mode",
    },
    {
        "question": "On considère la variable aléatoire X suivant la loi binomiale B(10, 0.3). Calculer P(X=3) et E(X).",
        "mode": "coaching",
        "desc": "Binomial distribution — coaching mode",
    },
]

In [ ]:
# ── Execute full RAG pipeline ────────────────────────────────

full_results = []

for i, q in enumerate(RAG_QUERIES, 1):
    print(f"\n{'='*75}")
    print(f"  FULL RAG QUERY {i}/{len(RAG_QUERIES)}: {q['desc']}")
    print(f"  Mode: {q['mode']}")
    print(f"  Q: {q['question']}")
    print(f"{'='*75}\n")

    result = engine.query(q["question"], mode=q["mode"])

    # ── Retrieval info ──
    print(f"  Retrieval case  : {result.retrieval_case}")
    print(f"  Confidence      : {result.confidence}")
    print(f"  Selected docs   : {len(result.selected_docs)}")
    print(f"  First pass docs : {len(result.first_pass_docs)}")
    print(f"  Second pass docs: {len(result.second_pass_docs)}")
    print(f"  Retrieval time  : {result.retrieval_time:.3f}s")
    print(f"  Generation time : {result.generation_time:.3f}s")
    print(f"  Total time      : {result.total_time:.3f}s")

    if result.error:
        print(f"  ERROR: {result.error}")

    # ── Retrieved sources ──
    print(f"\n  Sources used:")
    for d in result.selected_docs:
        m = d.metadata
        print(f"    Rank {d.rank} | dist={d.distance:.4f} | "
              f"type={m.get('type','')} | ch={m.get('chapter','')} | "
              f"yr={m.get('year','')} | sol={m.get('is_solution','')}")

    # ── Answer ──
    print(f"\n  {'─'*60}")
    print(f"  ANSWER:")
    print(f"  {'─'*60}")
    if result.answer:
        # Print full answer (may be long)
        for line in result.answer.split('\n'):
            print(f"  {line}")
    else:
        print(f"  (no answer — error: {result.error})")

    full_results.append({
        "idx": i,
        "desc": q["desc"],
        "mode": q["mode"],
        "case": result.retrieval_case,
        "confidence": result.confidence,
        "n_selected": len(result.selected_docs),
        "best_distance": round(result.selected_docs[0].distance, 4) if result.selected_docs else None,
        "retrieval_time": round(result.retrieval_time, 3),
        "generation_time": round(result.generation_time, 3),
        "total_time": round(result.total_time, 3),
        "has_answer": bool(result.answer),
        "error": result.error,
        "answer_length": len(result.answer) if result.answer else 0,
        "answer_preview": (result.answer or "")[:150].replace('\n', ' '),
    })

print(f"\n\nFull RAG tests complete: {len(full_results)} queries.")

### 4.1 Full RAG Summary Table

In [ ]:
hdr2 = (f"{'#':>2s}  {'Description':<35s}  {'Mode':<11s}  {'Case':>4s}  {'Conf':>6s}  "
        f"{'Docs':>4s}  {'BestD':>7s}  {'RetT':>5s}  {'GenT':>5s}  {'TotT':>5s}  "
        f"{'Ans':>5s}  {'Preview':<40s}")
sep2 = '-' * len(hdr2)

print(sep2)
print(hdr2)
print(sep2)

for r in full_results:
    desc_s = r['desc'][:33] + '..' if len(r['desc']) > 35 else r['desc']
    bd = f"{r['best_distance']:.4f}" if r['best_distance'] is not None else '  N/A '
    ok = 'OK' if r['has_answer'] else 'FAIL'
    preview = r['answer_preview'][:38] + '..' if len(r['answer_preview']) > 40 else r['answer_preview']
    print(f"{r['idx']:2d}  {desc_s:<35s}  {r['mode']:<11s}  {r['case']:>4s}  {r['confidence']:>6s}  "
          f"{r['n_selected']:4d}  {bd:>7s}  {r['retrieval_time']:5.2f}  {r['generation_time']:5.2f}  "
          f"{r['total_time']:5.2f}  {ok:>5s}  {preview:<40s}")

print(sep2)

successes = sum(1 for r in full_results if r['has_answer'])
print(f"\n  Success rate: {successes}/{len(full_results)}")
if full_results:
    avg_gen = sum(r['generation_time'] for r in full_results) / len(full_results)
    avg_tot = sum(r['total_time'] for r in full_results) / len(full_results)
    print(f"  Avg generation time: {avg_gen:.2f}s")
    print(f"  Avg total time     : {avg_tot:.2f}s")

---
## 5. Syllabus Guard Test

The system prompt forbids out-of-program methods (L'Hopital, Taylor series, etc.).  
Verify that the engine **refuses** and proposes a bac-compatible alternative.

In [ ]:
GUARD_QUERIES = [
    "Utilise la règle de L'Hôpital pour calculer lim(x->0) sin(x)/x",
    "Diagonalise la matrice A pour calculer A^n",
]

for gq in GUARD_QUERIES:
    print(f"\n{'='*75}")
    print(f"  SYLLABUS GUARD TEST")
    print(f"  Q: {gq}")
    print(f"{'='*75}\n")

    result = engine.query(gq, mode="correction")

    print(f"  Case: {result.retrieval_case} | Confidence: {result.confidence}")
    print(f"  Time: retrieval={result.retrieval_time:.2f}s generation={result.generation_time:.2f}s")
    print(f"\n  ANSWER (first 800 chars):")
    print(f"  {'─'*60}")
    answer_text = result.answer if result.answer else f"ERROR: {result.error}"
    for line in answer_text[:800].split('\n'):
        print(f"  {line}")
    if len(answer_text) > 800:
        print(f"  [...truncated, total {len(answer_text)} chars]")
    print()

---
## 6. Retrieval Edge Cases (FREE)

Test queries that are deliberately vague, off-topic, or in Derja  
to observe how the two-stage retrieval and confidence scoring behave.

In [ ]:
EDGE_QUERIES = [
    ("Kifeh na3mel exercice mte3 les nombres complexes?",
     "Derja input — should still retrieve complex number content"),
    ("What is the capital of France?",
     "Off-topic — should yield low confidence / high distance"),
    ("x",
     "Minimal input — stress test"),
    ("Bac 2015 exercice 3 nombres complexes correction",
     "Specific bac reference — should find exact match if indexed"),
]

for query, desc in EDGE_QUERIES:
    selected, fp, sp, case = engine._two_stage_retrieve(query)
    ctx = engine._build_context(selected)
    conf = engine._confidence_level(selected, ctx)
    bd = selected[0].distance if selected else None
    tc = selected[0].metadata.get('chapter', '') if selected else ''

    print(f"  [{desc}]")
    print(f"    Q: {query}")
    print(f"    Case={case}  Conf={conf}  Docs={len(selected)}  "
          f"BestDist={f'{bd:.4f}' if bd else 'N/A'}  TopCh={tc}")
    if selected:
        snip = selected[0].content[:80].replace('\n', ' ')
        print(f"    Top snippet: {snip}...")
    print()

---
## 7. Combined Results Export

Save all results to a JSON file for reproducibility and later analysis.

In [ ]:
import json
from datetime import datetime, timezone

export = {
    "metadata": {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "python_version": sys.version,
        "chunk_count": engine.chunk_count,
        "config": {
            "CHAT_MODEL_ID": config.CHAT_MODEL_ID,
            "EMBEDDING_MODEL": config.EMBEDDING_MODEL_NAME,
            "USE_FP16": config.USE_FP16,
            "RETRIEVE_K_FIRST_PASS": config.RETRIEVE_K_FIRST_PASS,
            "RETRIEVE_K_SECOND_PASS": config.RETRIEVE_K_SECOND_PASS,
            "USE_TOP_N": config.USE_TOP_N,
            "SIMILARITY_GOOD_THRESHOLD": config.SIMILARITY_GOOD_THRESHOLD,
            "SIMILARITY_FALLBACK_THRESHOLD": config.SIMILARITY_FALLBACK_THRESHOLD,
        },
    },
    "retrieval_only_results": retrieval_results,
    "full_rag_results": full_results,
}

out_path = "test_rag_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {out_path}")
print(f"File size: {os.path.getsize(out_path):,} bytes")

---
## 8. Final Validation Report

One-screen summary suitable for thesis screenshot.

In [ ]:
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║             TUNISIAN BAC MATH RAG — VALIDATION REPORT          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Environment                                                   ║
║    Python          : {sys.version.split()[0]:<42s} ║
║    Embedding model : {config.EMBEDDING_MODEL_NAME:<42s} ║
║    FP16            : {str(config.USE_FP16):<42s} ║
║    LLM             : {config.CHAT_MODEL_ID:<42s} ║
║    DB chunks       : {engine.chunk_count:<42,d} ║
║                                                                ║
║  Retrieval-only tests (FREE, no API cost)                      ║
║    Queries tested   : {len(retrieval_results):<41d} ║
║    Case A (match)   : {case_a:<41d} ║
║    Case B (fallback) : {case_b:<40d} ║
║    Avg best distance : {avg_dist:<40.4f} ║
║    Avg retrieval time: {avg_time:<40.3f} ║
║                                                                ║
║  Full RAG tests (with Gemini generation)                       ║
║    Queries tested  : {len(full_results):<41d} ║
║    Successes       : {successes:<41d} ║""")

if full_results:
    print(f"""║    Avg gen time    : {avg_gen:<41.2f} ║
║    Avg total time  : {avg_tot:<41.2f} ║""")

print(f"""║                                                                ║
║  Syllabus guard   : {len(GUARD_QUERIES)} queries tested                            ║
║  Edge cases       : {len(EDGE_QUERIES)} queries tested                            ║
║                                                                ║
║  Results exported to: test_rag_results.json                    ║
║                                                                ║
╚══════════════════════════════════════════════════════════════════╝
""")